In [1]:
import re
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from scipy.stats import kruskal
import scikit_posthocs as sp

import warnings
warnings.filterwarnings("ignore")

In [ ]:
cd C:\Github_repository\Metabolomics_Tutorial\Data

In [5]:
MA_PATH   = r"dataset_filtered_scaled_merged.xlsx"  #Edit: if you have a different dataset, change the filename here.
SHEET_NAME = r"fecal_neg_2w"  #Edit: if you have a different sheet name, change the sheet name here.

OUT_XLSX  = r"stats_" + SHEET_NAME + ".xlsx"

GROUP1 = 'Control'  #Edit: Control group
GROUP2 = 'VNAM'  #Edit: Treatment group
#GROUP3 = 'Solvent'
#GROUP4 = 'MIA'

In [6]:
# 1) MA_2w.csv 읽기 + 그룹행 파싱

ma = pd.read_excel(MA_PATH, sheet_name=SHEET_NAME)  #Edit

# 첫 컬럼명을 Label로 통일
ma = ma.rename(columns={ma.columns[0]: "Label"})

# 그룹행 찾기 (Group 또는 Label)
s = ma["Label"].astype(str).str.strip().str.lower()
grp_idx = ma.index[s.isin(["group", "label"])][0]

# sample -> group
grp_row = ma.loc[grp_idx, ma.columns[1:]].astype(str).str.strip()

def norm_group(g):
    u = g.upper()
    return {"GROUP1":"GROUP1",
            "GROUP2":"GROUP2",
            "GROUP3":"GROUP3",
            "GROUP4":"GROUP4",
            "QC":"QC",
            "blank":"blank"}.get(u, g)

sample2group = {c: norm_group(grp_row[c]) for c in grp_row.index}
g1_cols = [c for c,g in sample2group.items() if g==GROUP1]  #Edit
g2_cols = [c for c,g in sample2group.items() if g==GROUP2]  #Edit
#g3_cols = [c for c,g in sample2group.items() if g==GROUP3]  #Edit
#g4_cols = [c for c,g in sample2group.items() if g==GROUP4]  #Edit

print("grp_idx:", grp_idx, GROUP1 + ":", len(g1_cols), GROUP2 + ":", len(g2_cols))
#, GROUP3 + ":", len(g3_cols), GROUP4 + ":", len(g4_cols))  #Edit: if you have more than 2 groups, uncomment the above line and edit accordingly.
print(ma.head(3))

grp_idx: 0 Control: 11 VNAM: 10
              Label    C01_2w    C02_2w    C03_2w    C04_2w    C05_2w  \
0             Group   Control   Control   Control   Control   Control   
1   296.61697-8.708 -4.572774  0.223411  0.657662 -0.098263  0.496152   
2  487.23704-10.048  0.067086  0.453396  0.402793  0.660216  0.991129   

     C06_2w    C07_2w    C08_2w    C09_2w  ...    V01_2w    V02_2w    V03_2w  \
0   Control   Control   Control   Control  ...      VNAM      VNAM      VNAM   
1  0.948159  0.618992  0.542554  1.009734  ... -4.572774  1.106246  1.038808   
2  0.245269  0.808271 -1.357107   0.54334  ...  1.679011 -0.002437 -0.973854   

     V04_2w    V05_2w    V06_2w    V07_2w    V08_2w    V09_2w    V10_2w  
0      VNAM      VNAM      VNAM      VNAM      VNAM      VNAM      VNAM  
1  1.110776 -4.572774   1.00363     1.012  0.896026  0.766784  0.948976  
2 -1.232324  1.037276 -0.680447 -0.506565 -1.041567 -1.077987 -1.641813  

[3 rows x 22 columns]


In [7]:
# 2) 데이터 본문 만들기 (그룹행 삭제, QC/Blank 컬럼 삭제)

dat = ma.drop(index=grp_idx).copy()

# 여기서 label_col 쓰지 말고, 확정된 'Label'을 FeatureID로 변경
dat = dat.rename(columns={"Label": "FeatureID"})

use_cols = ["FeatureID"] + g1_cols + g2_cols
#+ g3_cols + g4_cols
dat = dat[use_cols].copy()

for c in g1_cols + g2_cols:
    # + g3_cols + g4_cols:
    dat[c] = pd.to_numeric(dat[c], errors="coerce")

dat

,FeatureID,C01_2w,C02_2w,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,...,V01_2w,V02_2w,V03_2w,V04_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w
1,296.61697-8.708,-4.572774,0.223411,0.657662,-0.098263,0.496152,0.948159,0.618992,0.542554,1.009734,...,-4.572774,1.106246,1.038808,1.110776,-4.572774,1.003630,1.012000,0.896026,0.766784,0.948976
2,487.23704-10.048,0.067086,0.453396,0.402793,0.660216,0.991129,0.245269,0.808271,-1.357107,0.543340,...,1.679011,-0.002437,-0.973854,-1.232324,1.037276,-0.680447,-0.506565,-1.041567,-1.077987,-1.641813
3,134.89469-16.398,1.110949,-0.553325,-1.168239,0.095419,-0.624860,0.859972,-0.493933,0.482891,-0.482458,...,0.392333,0.009938,-0.411548,0.119807,0.034094,0.289589,0.488096,0.136336,0.034401,0.110369
4,421.22647-16.397,-0.208549,-0.280528,-1.001226,0.724640,0.021564,0.014339,-0.897659,-0.042201,-0.645183,...,0.340197,-0.066963,-0.345408,0.776809,-0.229924,0.056632,0.812477,0.851845,0.826798,-0.304622
5,296.61713-13.231,-2.462796,-2.462796,-2.462796,-2.462796,-2.462796,1.835018,1.406076,-2.462796,1.875993,...,-2.462796,2.124917,1.984142,1.968177,-2.462796,2.026062,2.017696,1.789133,1.620294,1.880161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8565,520.83183-0.955,0.156101,0.534191,-1.894465,0.518628,-1.894465,-1.894465,-1.894465,-1.894465,-1.894465,...,0.229636,1.754321,0.106558,1.735748,1.456689,0.573400,1.517752,1.762983,0.845962,1.257555
8566,356.91342-0.925,0.725600,-0.122333,0.264382,-0.044398,-1.616397,0.118876,0.522011,0.205893,0.483042,...,-0.332511,0.921243,-0.469283,-0.615441,0.136369,-0.020823,-0.022763,-0.219156,-0.279917,0.444356
8567,127.00133-1,0.405881,0.729615,0.889764,0.451989,0.711235,-2.675664,-2.675664,0.989545,-2.675664,...,1.322063,0.890606,-2.675664,0.691364,0.114295,0.462780,0.280555,0.240864,1.081080,0.930098
8568,443.23-9.348,-1.493018,-0.778087,-0.536311,-0.465949,0.562218,0.266093,-0.747465,-0.525304,0.910013,...,-0.902029,0.865122,-0.136649,1.155890,-0.609713,-0.031706,0.029071,0.594503,1.098527,0.466718


In [ ]:
## iF only 2 Groups

# 3) long 포맷 + 유의성 검정

long = dat.melt(id_vars="FeatureID", var_name="Sample", value_name="Intensity")
long["Group"] = long["Sample"].map(sample2group)
# QC/BLANK는 이미 컬럼에서 제거됐지만, 방어적으로 한 번 더
long = long[long["Group"].isin([GROUP1, GROUP2])].dropna(subset=["Intensity"])
long["Intensity_for_test"] = long["Intensity"]

rows = []
for fid, sub in long.groupby("FeatureID", sort=False):
    # (1) 검정용 값
    g1 = sub.loc[sub["Group"]==GROUP1, "Intensity_for_test"].to_numpy(dtype=float)
    g2 = sub.loc[sub["Group"]==GROUP2,    "Intensity_for_test"].to_numpy(dtype=float)

    n_g1 = int(np.isfinite(g1).sum())
    n_g2 = int(np.isfinite(g2).sum())

    if n_g1 > 0 and n_g2 > 0:
        u_stat, p = mannwhitneyu(g1, g2, alternative="two-sided", method="auto")
        u_stat = float(u_stat); p = float(p)
    else:
        u_stat, p = np.nan, np.nan

    rows.append([fid, n_g1, n_g2, u_stat, p])

res = pd.DataFrame(rows, columns=[
    "FeatureID", GROUP1 + "_n", GROUP2 + "_n",
    "U_stat", "p-val"
])


# BH / BKY (패키지) 보정

alpha = 0.05  # Edit

p = res["p-val"].to_numpy(dtype=float)
ok = np.isfinite(p)

res["q-val_BH"]  = np.nan
res["q-val_BKY"] = np.nan

# BH
res.loc[ok, "q-val_BH"] = multipletests(p[ok], alpha=alpha, method="fdr_bh")[1]

# BKY (two-stage)
res.loc[ok, "q-val_BKY"] = multipletests(p[ok], alpha=alpha, method="fdr_tsbky")[1]

out = res.copy()
out

,FeatureID,Control_n,VNAM_n,U_stat,p-val,q-val_BH,q-val_BKY
0,296.61697-8.708,11,10,27.0,0.052500,0.067975,0.018358
1,487.23704-10.048,11,10,81.0,0.072549,0.090504,0.024442
2,134.89469-16.398,11,10,35.0,0.169706,0.197798,0.053419
3,421.22647-16.397,11,10,29.0,0.072549,0.090504,0.024442
4,296.61713-13.231,11,10,20.0,0.011407,0.017401,0.004700
...,...,...,...,...,...,...,...
8564,520.83183-0.955,11,10,8.0,0.000851,0.002666,0.000720
8565,356.91342-0.925,11,10,72.0,0.245278,0.277464,0.074934
8566,127.00133-1,11,10,41.5,0.358399,0.394746,0.106608
8567,443.23-9.348,11,10,36.0,0.192667,0.221963,0.059945


In [ ]:
## If more than 2 Groups

# =========================
# 3) long 포맷
# =========================

long = dat.melt(
    id_vars="FeatureID",
    var_name="Sample",
    value_name="Intensity"
)

long["Group"] = long["Sample"].map(sample2group)

# GROUP1~4만 사용
long = long[
    long["Group"].isin([GROUP1, GROUP2, GROUP3, GROUP4])
].dropna(subset=["Intensity"])

long["Intensity_for_test"] = long["Intensity"]


# =========================
# 4) Kruskal-Wallis test
# =========================

rows = []

for fid, sub in long.groupby("FeatureID", sort=False):

    g1 = sub.loc[
        sub["Group"] == GROUP1,
        "Intensity_for_test"
    ].to_numpy(dtype=float)

    g2 = sub.loc[
        sub["Group"] == GROUP2,
        "Intensity_for_test"
    ].to_numpy(dtype=float)

    g3 = sub.loc[
        sub["Group"] == GROUP3,
        "Intensity_for_test"
    ].to_numpy(dtype=float)

    g4 = sub.loc[
        sub["Group"] == GROUP4,
        "Intensity_for_test"
    ].to_numpy(dtype=float)

    # finite 값만 사용
    g1 = g1[np.isfinite(g1)]
    g2 = g2[np.isfinite(g2)]
    g3 = g3[np.isfinite(g3)]
    g4 = g4[np.isfinite(g4)]

    n_g1 = len(g1)
    n_g2 = len(g2)
    n_g3 = len(g3)
    n_g4 = len(g4)

    if n_g1 > 0 and n_g2 > 0 and n_g3 > 0 and n_g4 > 0:

        try:
            h_stat, p = kruskal(
                g1, g2, g3, g4,
                nan_policy="omit"
            )

            h_stat = float(h_stat)
            p = float(p)

        except ValueError:
            h_stat, p = np.nan, np.nan

    else:
        h_stat, p = np.nan, np.nan

    rows.append([
        fid,
        n_g1, n_g2, n_g3, n_g4,
        h_stat, p
    ])


res = pd.DataFrame(
    rows,
    columns=[
        "FeatureID",
        GROUP1 + "_n",
        GROUP2 + "_n",
        GROUP3 + "_n",
        GROUP4 + "_n",
        "H_stat",
        "p-val"
    ]
)

res

,FeatureID,Control_n,VNAM_n,Solvent_n,MIA_n,H_stat,p-val
0,61.03972-1.494,13,13,6,7,12.602592,0.005580
1,130.0651-11.471,13,13,6,7,6.458679,0.091306
2,190.0862-11.471,13,13,6,7,6.414258,0.093106
3,74.09648-0.565,13,13,6,7,7.080849,0.069365
4,74.09644-1.041,13,13,6,7,7.080849,0.069365
...,...,...,...,...,...,...,...
3670,102.0103-11.13,13,13,6,7,4.782941,0.188399
3671,86.02562-8.898,13,13,6,7,12.175637,0.006805
3672,221.11728-13.523,13,13,6,7,1.276921,0.734621
3673,391.30569-17.859,13,13,6,7,1.310494,0.726643


In [ ]:
## If more than 2 Groups

# =========================
# 5) Kruskal p-value
#    BH / BKY 보정
# =========================

alpha = 0.05

p = res["p-val"].to_numpy(dtype=float)
ok = np.isfinite(p)

res["q-val_BH"] = np.nan
res["q-val_BKY"] = np.nan

# BH
res.loc[ok, "q-val_BH"] = multipletests(
    p[ok],
    alpha=alpha,
    method="fdr_bh"
)[1]

# BKY
res.loc[ok, "q-val_BKY"] = multipletests(
    p[ok],
    alpha=alpha,
    method="fdr_tsbky"
)[1]


# =========================
# 6) Dunn's post-hoc test
# =========================

group_order = [
    GROUP1,
    GROUP2,
    GROUP3,
    GROUP4
]

pair_list = [
    (GROUP1, GROUP2),
    (GROUP1, GROUP3),
    (GROUP1, GROUP4),
    (GROUP2, GROUP3),
    (GROUP2, GROUP4),
    (GROUP3, GROUP4)
]


dunn_rows = []

for fid, sub in long.groupby("FeatureID", sort=False):

    sub_test = sub[
        ["Group", "Intensity_for_test"]
    ].copy()

    sub_test = sub_test[
        np.isfinite(sub_test["Intensity_for_test"])
    ]

    # 4개 그룹 모두 존재하는 feature만 Dunn test
    groups_present = set(sub_test["Group"].unique())

    if not all(g in groups_present for g in group_order):
        continue

    try:
        # 여기서는 일단 raw p-value 계산
        dunn_mat = sp.posthoc_dunn(
            sub_test,
            val_col="Intensity_for_test",
            group_col="Group",
            p_adjust=None
        )

        for gA, gB in pair_list:

            p_dunn = dunn_mat.loc[gA, gB]

            dunn_rows.append([
                fid,
                gA,
                gB,
                float(p_dunn)
            ])

    except ValueError:
        continue


dunn_res = pd.DataFrame(
    dunn_rows,
    columns=[
        "FeatureID",
        "Group1",
        "Group2",
        "Dunn_p"
    ]
)


# =========================
# 7) Dunn p-value
#    BH / BKY 보정
# =========================

p = dunn_res["Dunn_p"].to_numpy(dtype=float)
ok = np.isfinite(p)

dunn_res["Dunn_q_BH"] = np.nan
dunn_res["Dunn_q_BKY"] = np.nan

# 전체 Feature × pairwise comparison에 대해 BH
dunn_res.loc[ok, "Dunn_q_BH"] = multipletests(
    p[ok],
    alpha=alpha,
    method="fdr_bh"
)[1]

# 전체 Feature × pairwise comparison에 대해 BKY
dunn_res.loc[ok, "Dunn_q_BKY"] = multipletests(
    p[ok],
    alpha=alpha,
    method="fdr_tsbky"
)[1]


# =========================
# 8) Kruskal 결과와 Dunn 결과 합치기
# =========================

out = dunn_res.merge(
    res,
    on="FeatureID",
    how="left"
)

out

,FeatureID,Group1,Group2,Dunn_p,Dunn_q_BH,Dunn_q_BKY,Control_n,VNAM_n,Solvent_n,MIA_n,H_stat,p-val,q-val_BH,q-val_BKY
0,61.03972-1.494,Control,VNAM,0.024242,0.158503,0.155808,13,13,6,7,12.602592,0.005580,0.031645,0.025885
1,61.03972-1.494,Control,Solvent,0.142321,0.415975,0.408904,13,13,6,7,12.602592,0.005580,0.031645,0.025885
2,61.03972-1.494,Control,MIA,0.000596,0.017963,0.017658,13,13,6,7,12.602592,0.005580,0.031645,0.025885
3,61.03972-1.494,VNAM,Solvent,0.746306,0.894351,0.879147,13,13,6,7,12.602592,0.005580,0.031645,0.025885
4,61.03972-1.494,VNAM,MIA,0.121608,0.383647,0.377125,13,13,6,7,12.602592,0.005580,0.031645,0.025885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22045,102.07421-13.432,Control,Solvent,0.609278,0.829366,0.815267,13,13,6,7,0.882195,0.829722,0.860635,0.703999
22046,102.07421-13.432,Control,MIA,0.859468,0.946789,0.930693,13,13,6,7,0.882195,0.829722,0.860635,0.703999
22047,102.07421-13.432,VNAM,Solvent,0.358593,0.654063,0.642944,13,13,6,7,0.882195,0.829722,0.860635,0.703999
22048,102.07421-13.432,VNAM,MIA,0.801528,0.919786,0.904150,13,13,6,7,0.882195,0.829722,0.860635,0.703999


In [9]:
# 5) 저장 (통계결과 + (원하면) long)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    out.to_excel(w, index=False, sheet_name="stats")
    # 필요하면 다른 시트도 추가 가능
    # long.to_excel(w, index=False, sheet_name="long_for_plot")

print("Saved:", OUT_XLSX, "shape=", out.shape)

Saved: stats_fecal_neg_2w.xlsx shape= (8569, 7)
